# Full-pool reverse ANN retrieval benchmark

Run in Colab CPU with high RAM if available. Put `colab_reverse_bundle.zip` and the original four training TSVs in Google Drive:

```
MyDrive/ml_challenge/colab_reverse_bundle.zip
MyDrive/ml_challenge/train/train_source1.tsv
MyDrive/ml_challenge/train/train_source2.tsv
MyDrive/ml_challenge/train/train_source3.tsv
MyDrive/ml_challenge/train/train_ground_truth.tsv
```

The bundle contains only code and the existing development retrieval cohort. The ANN index and target scan use no ground-truth labels. The first run is a speed profile, **not** a full-pool recall measurement. The final cell compares raw and K-capped retrieval on the 800 development S1 entities only. It never opens the previously used 200-entity confirmation partition.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, zipfile, subprocess, time, json, shutil
DRIVE = Path('/content/drive/MyDrive/ml_challenge')
DATA = DRIVE / 'train'
BUNDLE = DRIVE / 'colab_reverse_bundle.zip'
for p in [BUNDLE, *(DATA / f'train_source{i}.tsv' for i in (1,2,3)), DATA / 'train_ground_truth.tsv']:
    assert p.exists(), f'Missing {p}'
ROOT = Path('/content/ml-challenge')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(ROOT)
print('Bundle ready:', ROOT)


In [ ]:
# CPU FAISS; a GPU runtime is not required.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faiss-cpu==1.15.1', 'scikit-learn', 'pyarrow', 'unidecode',
                'lightgbm', 'rapidfuzz'], check=True)
import faiss, pandas as pd, numpy as np
print('FAISS', faiss.__version__, 'available RAM GB',
      round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1))


In [ ]:
PIPELINE = ROOT / 'code/business_entity_resolution/01_pipeline.py'
BASE = ROOT / 'research_runs/ber_fullpool_1000_offset300_aug1'
CACHE = ROOT / 'research_runs/reverse_ann_cache'
PROFILE = ROOT / 'research_runs/reverse_ann_profile'
FULL = ROOT / 'research_runs/reverse_ann_full'
COMMON = [sys.executable, str(PIPELINE), '--split', 'train',
          '--data-dir', str(DRIVE), '--reverse-ann-from', str(BASE),
          '--ann-cache-dir', str(CACHE), '--reverse-top', '10',
          '--cap', '150', '--target-chunk', '20000',
          '--reverse-source-chunk', '50000', '--ann-components', '128',
          '--ann-nlist', '2048', '--ann-nprobe', '16', '--ann-threads', '8']
started = time.monotonic()
subprocess.run(COMMON + ['--output-dir', str(PROFILE),
                         '--ann-profile-targets', '100000'], check=True)
elapsed = time.monotonic() - started
population = 10320219
print(f'Profile wall time (includes building full S1 index): {elapsed/60:.1f} min')
print('Cache:', CACHE, 'size GB',
      round(sum(p.stat().st_size for p in CACHE.rglob('*') if p.is_file()) / 1e9, 2))


In [ ]:
# This second bounded run reuses the index and measures target scanning alone.
started = time.monotonic()
subprocess.run(COMMON + ['--output-dir', str(PROFILE),
                         '--ann-profile-targets', '100000'], check=True)
scan_seconds = time.monotonic() - started
projected_hours = scan_seconds * population / 100000 / 3600
print(f'100k-target scan: {scan_seconds:.1f} s; projected full scan: {projected_hours:.1f} h')
print('Projection is approximate and excludes Drive output/copy time.')


In [ ]:
# Run only when the measured runtime fits your Colab session.
MAX_PROJECTED_HOURS = 10
assert projected_hours <= MAX_PROJECTED_HOURS, (
    f'Projected {projected_hours:.1f} h exceeds the {MAX_PROJECTED_HOURS} h session guard. '
    'Use a larger CPU runtime or revise the ANN implementation before claiming full-pool recall.')
started = time.monotonic()
subprocess.run(COMMON + ['--output-dir', str(FULL)], check=True)
print('Full run hours:', round((time.monotonic() - started)/3600, 2))
manifest = json.loads((FULL / 'train_retrieval_run.json').read_text())
assert manifest['target_pool_size'] == 10320219, manifest
assert manifest['labels_used_for_retrieval'] is False
assert manifest['ann_profile_targets'] is None
print(json.dumps(manifest, indent=2))


In [ ]:
# Score blocker retrieval on the 800 development entities only.
# The 200 previously opened confirmation entities are excluded.
import importlib.util
spec = importlib.util.spec_from_file_location('validate', ROOT / 'code/business_entity_resolution/03_validate.py')
validate = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate)
source = pd.read_parquet(BASE / 'train_s1.parquet')
design, development, previously_opened = validate.sealed_partitions(source)
dev_ids = design | development
assert len(dev_ids) == 800 and len(previously_opened) == 200
truth = validate.match.parse_truth(DATA / 'train_ground_truth.tsv', dev_ids)
before = pd.read_parquet(BASE / 'train_route_audit.parquet')
after = pd.read_parquet(FULL / 'train_route_audit.parquet')
routes = list(validate.match.ROUTES) + ['reverse']
def recall_table(audit, routes):
    audit = audit[audit.source1_entity_id.isin(dev_ids)].copy()
    grouped = {sid: f for sid, f in audit.groupby('source1_entity_id', sort=False)}
    totals = {k: 0 for k in (40,60,80,100,150,'raw')}
    truth_count = sum(len(truth.get(sid,set())) for sid in dev_ids)
    for sid in dev_ids:
        f = grouped.get(sid)
        if f is None:
            continue
        ranks = f[[f'{r}_rank' for r in routes]].fillna(0).to_numpy()
        active = ranks > 0
        agree = active.sum(axis=1)
        reciprocal = np.where(active, 1/(ranks+1), 0).sum(axis=1)
        ordered = sorted(range(len(f)), key=lambda j: (
            -agree[j], -reciprocal[j], f.candidate_entity_id.iat[j]))
        ids = f.candidate_entity_id.iloc[ordered].tolist()
        true = truth.get(sid,set())
        totals['raw'] += len(true & set(ids))
        for k in (40,60,80,100,150):
            totals[k] += len(true & set(ids[:k]))
    return {'S1':len(dev_ids), 'true_links':truth_count, 'candidate_pairs':len(audit),
            **{str(k): round(v/truth_count,6) for k,v in totals.items()}}
report = {'forward': recall_table(before, list(validate.match.ROUTES)),
          'forward_plus_reverse_ann': recall_table(after, routes)}
print(json.dumps(report, indent=2))
Path(FULL / 'development_retrieval_report.json').write_text(json.dumps(report, indent=2))


In [ ]:
# Save the report and candidates back to Drive for local matcher work.
DEST = DRIVE / 'reverse_ann_result'
DEST.mkdir(exist_ok=True)
for filename in ('train_s1.parquet','train_target.parquet','train_pairs.parquet',
                 'train_route_audit.parquet','train_retrieval_run.json',
                 'development_retrieval_report.json'):
    shutil.copy2(FULL / filename, DEST / filename)
print('Saved', DEST)
